In [ ]:
from libero.libero import benchmark
from libero.libero import get_libero_path
from libero.libero.envs import OffScreenRenderEnv
from openpi_client import image_tools

import pathlib
import numpy as np
import math
import os

CAMERA_NAMES = ["agentview", "birdview", "robot0_eye_in_hand", "sideview", "canonical_frontview"]

def _get_libero_env(task, resolution, seed):
    """Initializes and returns the LIBERO environment, along with the task description."""
    task_description = task.language
    task_bddl_file = pathlib.Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env_args = {"bddl_file_name": task_bddl_file, "camera_heights": resolution, "camera_widths": resolution, "camera_names": CAMERA_NAMES}
    env = OffScreenRenderEnv(**env_args)
    env.seed(seed)  # IMPORTANT: seed seems to affect object positions even when using fixed initial state
    return env, task_description

def _get_empty_env(task, env):
    import robosuite
    dataset_file = os.path.join(get_libero_path("datasets"), f"{task.problem_folder}/{task.name}_demo.hdf5")
    import h5py
    import json
    f = h5py.File(dataset_file, "r")
    env_meta = json.loads(f["data"].attrs["env_args"])
    f.close()
    empty_env_kwargs = env_meta['env_kwargs'].copy()
    empty_env_kwargs['env_name'] = "SingleArmEmptyEnv"
    empty_env_kwargs['hard_reset'] = False
    empty_env_kwargs['ignore_done'] = True
    empty_env_kwargs['has_offscreen_renderer'] = False
    empty_env_kwargs['has_renderer'] = False
    empty_env_kwargs['use_camera_obs'] = False
    empty_env_kwargs['camera_names'] = CAMERA_NAMES
    empty_env_kwargs['camera_heights'] = LIBERO_ENV_RESOLUTION
    empty_env_kwargs['camera_widths'] = LIBERO_ENV_RESOLUTION
    empty_env_kwargs['robots'] = [type(robot.robot_model).__name__ for robot in env.robots]
    empty_env = robosuite.make(**empty_env_kwargs)
    empty_env.copy_env_model(env)
    return empty_env


task_suite_name = "libero_object"
task_id = 7
seed = 8
from vlm_agent import VLMAgent
from vlm_utils import *
# agent = VLMAgent(task_suite_name, task_id)

LIBERO_ENV_RESOLUTION = 224
LIBERO_DUMMY_ACTION = [0.0] * 6 + [-1.0]

benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[task_suite_name]()
task = task_suite.get_task(task_id)
initial_states = task_suite.get_task_init_states(task_id)
env, task_description = _get_libero_env(task, LIBERO_ENV_RESOLUTION, seed)
empty_env = _get_empty_env(task, env)
print(f"Task description: {task_description}")

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


Task description: pick up the ketchup and place it in the basket


In [4]:
env.reset()
for _ in range(10):
    obs, reward, done, info = env.step(LIBERO_DUMMY_ACTION)
from PIL import Image
Image.fromarray(obs["agentview_image"][::-1]).save("agentview_image.png")
Image.fromarray(obs["birdview_image"][::-1]).save("birdview_image.png")
Image.fromarray(obs['sideview_image'][::-1]).save("sideview_image.png")

In [10]:
from wm_client.client import WMClient
from wm_client.wm_env import WMEnv
host = "0.0.0.0"
port = 7880
wm_client = WMClient(host, port)

Connecting to ws://0.0.0.0:7880...
Connected to ws://0.0.0.0:7880


In [11]:
from dp_utils import embed_lang
avail_task_suite = benchmark_dict["libero_object"]()
subtask_id = task_id
subtask = avail_task_suite.get_task(subtask_id)
subtask_description = subtask.language.replace(" up", "")
# subtask_description = subtask.language.replace("place", "put")
# subtask_description = subtask_description.replace("cheese", "cheese box")
print(f"Subtask description: {subtask_description}")
subtask_embedding = embed_lang(subtask_description)

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Subtask description: pick the alphabet soup and place it in the basket
Loaded language embed from cache.


In [12]:
wm_env = WMEnv(env, empty_env, wm_client)
wm_env.reset()
num_steps = 0
replay_images = []
obs = env.reset()
# obs = env.set_init_state(initial_states[seed])
for t in range(80):
    obs, reward, done, info = wm_env.step(LIBERO_DUMMY_ACTION)
agent.start_episode(obs)

/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: divide by zero encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: invalid value encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:161: RuntimeWarning: invalid value encountered in cast
  pixels = pixels[..., :2].round().astype(int)  # shape [..., 2]


In [13]:
from dp_utils import load_checkpoint
import robosuite.utils.transform_utils as T
# checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.20/05.27.36_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0075-test_mean_score=1.000.ckpt"
# checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.02.27/20.14.00_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0022-test_mean_score=1.000.ckpt"
# checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.21/22.48.59_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0035-test_mean_score=0.900.ckpt"
# checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.20/06.20.27_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0540-test_mean_score=0.800.ckpt"
checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.14/08.49.11_train_diffusion_transformer_hybrid_libero_image/checkpoints/epoch=0460-test_mean_score=0.100.ckpt"
policy, cfg = load_checkpoint(checkpoint_path)
policy = policy.to("cuda")
import torch
def to_torch(image):
    image = image_tools.resize_with_pad(image, 128, 128)
    return np.moveaxis(image[::-1], -1, 0) / 255.0
def policy_fn(obs):
    np_obs_dict = dict(obs)
    if "lang_embed" in cfg.shape_meta.obs:
        np_obs_dict["lang_embed"] = subtask_embedding
    obs_keys = cfg.shape_meta.obs.keys()
    np_obs_dict = {k: np_obs_dict[k] for k in obs_keys}
    np_obs_dict = {k: to_torch(v) if "image" in k else v for k, v in np_obs_dict.items()}
    obs_dict = {k: torch.from_numpy(v).to("cuda").unsqueeze(0).unsqueeze(0) for k, v in np_obs_dict.items()}
    with torch.no_grad():
        action_dict = policy.predict_action(obs_dict)
    np_action_dict = {k: v.cpu().numpy() for k, v in action_dict.items()}
    action = np_action_dict['action_pred'][0]
    return action

idm_checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.17/08.56.15_train_diffusion_unet_lowdim_idm_libero_idm/checkpoints/epoch=0110-val_loss=0.019.ckpt"
# idm_checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.12/01.36.19_train_diffusion_unet_lowdim_idm_libero_idm/checkpoints/epoch=0020-val_loss=0.025.ckpt"
idm, idm_cfg = load_checkpoint(idm_checkpoint_path)
idm = idm.to("cuda")
def idm_fn(obs, target_pos, target_quat=None):
    np_obs_dict = dict(obs)
    obs_keys = idm_cfg.shape_meta.obs.keys()
    np_obs_dict = {k: np_obs_dict[k] for k in obs_keys}
    delta_obs_dict = {"robot0_eef_pos": target_pos - obs['robot0_eef_pos']}
    if target_quat is not None:
        delta_obs_dict['robot0_eef_quat'] = T.quat_distance(target_quat, obs['robot0_eef_quat'])
    obs_dict = {k: torch.from_numpy(v).to("cuda").unsqueeze(0) for k, v in np_obs_dict.items()}
    delta_obs_dict = {k: torch.from_numpy(v).to("cuda").unsqueeze(0) for k, v in delta_obs_dict.items()}
    with torch.no_grad():
        action_dict = idm.predict_action(obs_dict, delta_obs_dict)
    np_pred_action = action_dict['action_pred'].cpu().numpy()[0]
    return np_pred_action

idm_2_checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.17/23.50.14_train_diffusion_unet_lowdim_idm_libero_idm/checkpoints/epoch=0060-val_loss=0.020.ckpt"
# idm_2_checkpoint_path = "/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/diffusion_policy/data/outputs/2026.03.11/21.52.52_train_diffusion_unet_lowdim_idm_libero_idm/checkpoints/epoch=0040-val_loss=0.026.ckpt"
idm_2, idm_2_cfg = load_checkpoint(idm_2_checkpoint_path)
idm_2 = idm_2.to("cuda")
def idm_fn_2(obs, target_pos, target_quat=None):
    np_obs_dict = dict(obs)
    obs_keys = idm_2_cfg.shape_meta.obs.keys()
    np_obs_dict = {k: np_obs_dict[k] for k in obs_keys}
    delta_obs_dict = {"robot0_eef_pos": target_pos - obs['robot0_eef_pos']}
    if target_quat is not None:
        delta_obs_dict['robot0_eef_quat'] = T.quat_distance(target_quat, obs['robot0_eef_quat'])
    obs_dict = {k: torch.from_numpy(v).to("cuda").unsqueeze(0) for k, v in np_obs_dict.items()}
    delta_obs_dict = {k: torch.from_numpy(v).to("cuda").unsqueeze(0) for k, v in delta_obs_dict.items()}
    with torch.no_grad():
        action_dict = idm_2.predict_action(obs_dict, delta_obs_dict)
    np_pred_action = action_dict['action_pred'].cpu().numpy()[0]
    return np_pred_action


============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['robot0_gripper_qpos', 'robot0_eef_pos', 'lang_embed', 'robot0_eef_quat']
using obs modality: rgb with keys: ['agentview_image', 'robot0_eye_in_hand_image']
using obs modality: depth with keys: []
using obs modality: scan with keys: []


In [17]:
import tqdm
pbar = tqdm.tqdm(total=500, desc="Executing policy")
while not done and num_steps < 160:
    action_chunk = policy_fn(obs)[:10]
    for action in (action_chunk):
        obs, reward, done, info = wm_env.step(action)
        replay_images.append(obs["agentview_image"][::-1])
    pbar.update(10)
    num_steps += 10
        
    

Executing policy:  32%|█████████████                            | 160/500 [00:16<00:35,  9.55it/s]

In [14]:
target_object_position = agent.identify_target_object()
# target_object_position = agent.get_action_proposal()  
plot_coordinates_on_image(obs, target_object_position)
target_point = generate_3d_point(target_object_position, empty_env.get_camera_info())
action_chunk = update_gripper_action(idm_fn(obs, target_point), -1)

Found 1 objects
Found 0 objects
No object found in the topview image for the prompt: a blue round can
Found 1 objects


In [7]:
obs

OrderedDict([('robot0_joint_pos',
              array([ 1.29000557e-12, -1.61037389e-01,  3.39744265e-12, -2.44459747e+00,
                      1.60572090e-12,  2.22675220e+00,  7.85398163e-01])),
             ('robot0_joint_pos_cos',
              array([ 1.        ,  0.98706148,  1.        , -0.76677449,  1.        ,
                     -0.60991702,  0.70710678])),
             ('robot0_joint_pos_sin',
              array([ 1.29000557e-12, -1.60342259e-01,  3.39744265e-12, -6.41916572e-01,
                      1.60572090e-12,  7.92465287e-01,  7.07106781e-01])),
             ('robot0_joint_vel',
              array([-3.96942141e-14, -2.59343404e-17, -7.33697046e-14,  6.08684367e-17,
                     -7.15437454e-14,  4.84270371e-17,  2.95041473e-13])),
             ('robot0_eef_pos',
              array([-1.48464661e-01,  2.23052204e-12,  2.61279476e-01])),
             ('robot0_eef_quat',
              array([ 9.99596605e-01,  2.46212833e-04, -2.84001205e-02, -6.99529622e-06]

In [14]:
target_point = np.array([0.15090574, 0.03021878, 0.03839291])
target_point += np.array([0, 0, 0.1])
target_point += np.array([0.0, 0.02, 0])
action_chunk = update_gripper_action(idm_fn(obs, target_point), -1)

In [10]:
for action in action_chunk[:20]:
    obs, reward, done, info = wm_env.step(action)
    replay_images.append(obs["agentview_image"][::-1])
    num_steps += 1
action_chunk = update_gripper_action(idm_fn_2(obs, target_point), -1)

/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: divide by zero encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:160: RuntimeWarning: invalid value encountered in divide
  pixels = pixels / pixels[..., 2:3]
/net/holy-isilon/ifs/rc_labs/ydu_lab/xczhang/workspace/SAILOR/env_repos/robosuite/robosuite/utils/camera_utils.py:161: RuntimeWarning: invalid value encountered in cast
  pixels = pixels[..., :2].round().astype(int)  # shape [..., 2]


In [12]:
import imageio
with wm_env.simulation():
    pred_obs = wm_env.simulate(action_chunk)
    wm_agent_obs = pred_obs['future_obs']
imageio.mimwrite('test_dp_output_wm_1.mp4', pred_obs['WMPredictionOutput'].full_video, fps=20)

In [ ]:
endpoint_response = agent.optimize_endpoint(wm_agent_obs)

In [ ]:
target_point += optimize_endpoint(endpoint_response, scale=0.08)

In [ ]:
height_response = agent.optimize_height_sideview(wm_agent_obs)

In [ ]:
target_point += np.array([0, 0, height_response]) * 0.1

In [ ]:
candidate_points = generate_candidates(target_point, scale=0.05)

In [ ]:
import imageio
candidate_obs = []
candidate_actions = []
for i, candidate_point in enumerate(candidate_points[:1]):
    with wm_env.simulation():
        # candidate_action = update_gripper_action(idm_fn_2(obs, candidate_point, target_quat), gripper_action)
        candidate_action = action_chunk
        candidate_actions.append(candidate_action)
        next_obs = wm_env.simulate(candidate_action)
        # next_action_chunk = policy_fn(next_obs['future_obs'][-1])[:20]
        # next_obs = wm_env.simulate(next_action_chunk)
        # next_action_chunk = policy_fn(next_obs['future_obs'][-1])[:20]
        # next_obs = wm_env.simulate(next_action_chunk)
    imageio.mimwrite(f'test_dp_output_candidate{i}.mp4', next_obs['WMPredictionOutput'].full_video, fps=20)
    candidate_obs.append(next_obs['future_obs'])

In [ ]:
# frontview_ranking = agent.rank_images_frontview(candidate_obs)
wristview_ranking = agent.rank_images_wristview(candidate_obs)

In [ ]:
action_chunk = candidate_actions[wristview_ranking[0]]
target_point = candidate_points[wristview_ranking[0]]

In [15]:
for action in action_chunk:
    obs, reward, done, info = wm_env.step(action)
    replay_images.append(obs["agentview_image"][::-1])

In [18]:
import imageio
imageio.mimwrite('test_dp_output.mp4', replay_images, fps=20)
replay_images = []